# MODE 2 — M2_F06 AIRCRAFT CARRIER — CONTROL

> Phase 8 — Dual Pipeline Doctrine — v1.0.0

| Loi | Règle |
|-----|-------|
| R-01 | Isolation Mode 2 |
| R-04 | Overlay binaire : OUI (audio+texte) / NON (vidéo brute) |

In [ ]:
from pathlib import Path
import subprocess, shutil, json

FREGATE_ROOT = Path("../")
dirs = {
    "IN_FINAL_FRAMES": FREGATE_ROOT / "IN_FINAL_FRAMES",
    "IN_AUDIO":        FREGATE_ROOT / "IN_AUDIO",
    "OUT_FINAL_MOVIE": FREGATE_ROOT / "OUT_FINAL_MOVIE",
    "OUT_REPORT":      FREGATE_ROOT / "OUT_REPORT",
}
print("=== M2_F06 CONTROL — PRÉ-VOL ===")
for name, d in dirs.items():
    icon = "✅" if d.exists() else "❌"
    print(f"  {icon} {name}")

In [ ]:
# Frames disponibles
frames_dir = dirs["IN_FINAL_FRAMES"]
frames = sorted([f for f in frames_dir.iterdir() if f.suffix.lower() in (".png", ".exr")]) if frames_dir.exists() else []
print(f"Frames disponibles : {len(frames)}")
if frames:
    print(f"  Premier : {frames[0].name}")
    print(f"  Dernier : {frames[-1].name}")

# Audio
audio_files = list(dirs["IN_AUDIO"].glob("*")) if dirs["IN_AUDIO"].exists() else []
audio_files = [f for f in audio_files if f.suffix.lower() in (".wav",".mp3",".ogg",".aac",".flac")]
print(f"Audio : {len(audio_files)} fichier(s)")
for a in audio_files:
    print(f"  📢 {a.name}")

In [ ]:
# Vérification outils
print("=== OUTILS ===")
for tool in ["ffmpeg", "rife-ncnn-vulkan", "realcugan-ncnn-vulkan"]:
    path = shutil.which(tool)
    icon = "✅" if path else "⚠️ "
    label = path if path else "non trouvé dans PATH"
    print(f"  {icon} {tool} : {label}")

In [ ]:
# Lecture rapport si existant
report_path = dirs["OUT_REPORT"] / "m2_f06_report.json"
if report_path.exists():
    with open(report_path) as f:
        r = json.load(f)
    icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"{icon} STATUS : {r['status']} ({r['timestamp']})")
    print(f"   Overlay       : {r.get('overlay_mode')}")
    for step, data in r.get("steps", {}).items():
        ok = "✅" if data.get("ok") else "❌"
        skip = " (SKIPPED)" if data.get("skipped") else ""
        print(f"   {ok} {step}{skip}")
    out = r.get("outputs", {})
    if out.get("final"):
        fp = Path(out["final"])
        size = fp.stat().st_size/(1024*1024) if fp.exists() else 0
        print(f"   Output final  : {fp.name} ({size:.1f} MB)")
else:
    print("Aucun rapport — lancer la production d'abord")